# 🎬 Rufus — YouTube Automation Pipeline

**One click per cell. Videos go to Google Drive automatically.**

---
### Before you start:
1. `Runtime → Change runtime type → GPU → L4` (or A100)
2. Run cells **in order**, top to bottom
3. Cell 3 asks for your API keys — paste them in


In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Check GPU
# ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Mount Google Drive (videos saved here)
# ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/Rufus/output'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"✅ Videos will be saved to: {DRIVE_OUTPUT}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — API Keys (paste yours here)
# ─────────────────────────────────────────────────────────
import os

# Required: free key at pexels.com/api
os.environ['PEXELS_API_KEY'] = 'PASTE_YOUR_PEXELS_KEY_HERE'

# Optional: YouTube upload (leave empty to skip upload)
os.environ['YOUTUBE_CLIENT_SECRETS_FILE'] = ''

# Output path → Google Drive
os.environ['OUTPUT_PATH'] = DRIVE_OUTPUT
os.environ['QDRANT_HOST'] = ''  # use FAISS (no Docker needed)

print("✅ Keys set")
print(f"   Pexels: {'✅ SET' if os.environ.get('PEXELS_API_KEY','').startswith('PASTE') == False else '❌ NOT SET — paste your key!'}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Install system dependencies
# ─────────────────────────────────────────────────────────
print("Installing FFmpeg...")
!apt-get install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version 2>&1 | head -1

print("\nInstalling Ollama...")
!curl -fsSL https://ollama.ai/install.sh | sh > /dev/null 2>&1
print("✅ Ollama installed")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Clone Rufus
# ─────────────────────────────────────────────────────────
import os

RUFUS_DIR = '/content/Rufus'

if os.path.exists(RUFUS_DIR):
    print("Rufus already cloned — pulling latest...")
    !cd {RUFUS_DIR} && git pull origin claude/yt-viral-automation-xc6h3
else:
    print("Cloning Rufus...")
    !git clone https://github.com/MisterRufus-code/Rufus.git {RUFUS_DIR}
    !cd {RUFUS_DIR} && git checkout claude/yt-viral-automation-xc6h3

os.chdir(RUFUS_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Install Python packages (takes ~3 minutes)
# ─────────────────────────────────────────────────────────
print("Installing Python packages...")
!pip install -q -r requirements.txt
!pip install -q pyyaml python-dotenv
print("\n✅ All packages installed")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Start Ollama + Pull Model
# Choose your model:
#   llama3.1        — fast, good quality (4.7GB)
#   llama3.1:70b    — best quality (40GB) — needs A100
#   deepseek-r1:32b — excellent (20GB) — needs L4/A100
#   gemma3:27b      — great (17GB) — works on L4
# ─────────────────────────────────────────────────────────
import subprocess, time, httpx

MODEL = 'llama3.1'  # ← Change this to use a bigger model

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull model
print(f"Pulling {MODEL} (this may take a few minutes)...")
!ollama pull {MODEL}

# Verify
try:
    r = httpx.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f"\n✅ Ollama running — models: {models}")
except Exception as e:
    print(f"❌ Ollama not responding: {e}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Update config to use selected model
# ─────────────────────────────────────────────────────────
import yaml
from pathlib import Path

NICHE = 'dark_triad_ai'  # ← Change niche here if needed

niche_path = Path(f'config/niches/{NICHE}.yaml')
data = yaml.safe_load(niche_path.read_text())
data['model'] = MODEL
niche_path.write_text(yaml.dump(data, allow_unicode=True, default_flow_style=False))

print(f"✅ Niche '{NICHE}' updated to use model: {MODEL}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — Create .env file
# ─────────────────────────────────────────────────────────
env_content = f"""PEXELS_API_KEY={os.environ.get('PEXELS_API_KEY', '')}
OUTPUT_PATH={DRIVE_OUTPUT}
QDRANT_HOST=
RUFUS_LOG_LEVEL=INFO
"""
Path('.env').write_text(env_content)
print("✅ .env created")
print(env_content)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10 — 🚀 RUN THE PIPELINE
# This generates a full video and saves to Google Drive
# Takes ~15-25 minutes on L4, ~8-12 minutes on A100
# ─────────────────────────────────────────────────────────
import os
os.chdir('/content/Rufus')

!python main.py pipeline --niche {NICHE}

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 11 — Preview video in notebook
# ─────────────────────────────────────────────────────────
from pathlib import Path
from IPython.display import Video, display
import glob

# Find latest video
videos = sorted(glob.glob(f'{DRIVE_OUTPUT}/**/final_video.mp4', recursive=True))
if not videos:
    videos = sorted(glob.glob(f'{DRIVE_OUTPUT}/**/*.mp4', recursive=True))

if videos:
    latest = videos[-1]
    print(f"\n✅ Video: {latest}")
    display(Video(latest, embed=True, width=800))
else:
    print("❌ No video found — check output above for errors")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 12 — Read latest script
# ─────────────────────────────────────────────────────────
import glob
scripts = sorted(glob.glob('logs/scripts/*.txt'))
if scripts:
    print(open(scripts[-1]).read())
else:
    print("No script logs found")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 13 — Run pipeline again (different topic)
# Just run this cell repeatedly for more videos
# ─────────────────────────────────────────────────────────
import os
os.chdir('/content/Rufus')
!python main.py pipeline --niche {NICHE}

---
## 💡 Tips

**Change model** → Edit `MODEL` in Cell 7 and re-run Cell 7 + 8
- `llama3.1` — fast, works on T4
- `gemma3:27b` — better quality, needs L4
- `deepseek-r1:32b` — best quality, needs L4/A100
- `llama3.1:70b` — maximum quality, needs A100

**Run multiple videos** → Just keep running Cell 13

**Find videos** → Google Drive → MyDrive → Rufus → output

**Session expired?** → Run cells 1, 5, 7, 9, 10 (skip install cells)

**Keep session alive** → Paste in browser console (F12):
```javascript
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000)
```
